# DFT расчёт HEA L12 (32 атома) на GPU — PySCF + gpu4pyscf

**Важно:**
1. **Среда выполнения -> Сменить тип -> GPU**.
2. Загрузи `HEA_L12_SQS_32atoms.cif`.

gpu4pyscf — официальный GPU-плагин PySCF.

In [ ]:
!pip install pyscf cupy-cuda12x ase
!pip install gpu4pyscf-cuda12x

import cupy
print(f"GPU найден: {cupy.cuda.runtime.getDeviceCount()} устройств")
print(f"PySCF: {__import__('pyscf').__version__}")
import gpu4pyscf
print(f"gpu4pyscf: {gpu4pyscf.__version__}")

In [ ]:
from ase.io import read, write
from ase.optimize import BFGS
from ase.calculators.calculator import Calculator, all_changes
try:
    from ase.filters import FrechetCellFilter as CellFilter
except ImportError:
    try:
        from ase.filters import ExpCellFilter as CellFilter
    except ImportError:
        from ase.constraints import ExpCellFilter as CellFilter

from pyscf.pbc import gto, dft
import numpy as np
import os

# GTH базисы MOLOPT-SR содержат 3d металлы
import requests
basis_url = 'https://raw.githubusercontent.com/pyscf/pyscf/master/pyscf/pbc/gto/basis/gth-szv-molopt-sr.dat'
if not os.path.exists('gth-szv-molopt-sr.dat'):
    print('Скачивание gth-szv-molopt-sr.dat...')
    r = requests.get(basis_url)
    r.raise_for_status()
    with open('gth-szv-molopt-sr.dat', 'w') as f:
        f.write(r.text)
    print(f'  Загружено: {os.path.getsize("gth-szv-molopt-sr.dat")} байт')
else:
    print('  gth-szv-molopt-sr.dat уже есть')

basis_name = 'gth-szv-molopt-sr.dat'
pseudo_name = 'gth-pbe'
print(f"Базис: {basis_name} + {pseudo_name}")

HARTREE_TO_EV = 27.211386245988
BOHR_TO_ANG = 0.529177210903
HA_BOHR3_TO_EV_ANG3 = HARTREE_TO_EV / (BOHR_TO_ANG**3)

class PySCFGPUCalculator(Calculator):
    implemented_properties = ['energy', 'forces', 'stress']

    def __init__(self, xc='PBE', kpts=(1, 1, 1), **kwargs):
        Calculator.__init__(self, **kwargs)
        self.xc = xc
        self.kpts = kpts

    def calculate(self, atoms=None, properties=['energy'], system_changes=all_changes):
        Calculator.calculate(self, atoms, properties, system_changes)

        cell = gto.Cell()
        symbols = atoms.get_chemical_symbols()
        positions = atoms.get_positions()
        cell.atom = [[symbols[i], positions[i].tolist()] for i in range(len(atoms))]
        cell.a = atoms.get_cell()[:]
        cell.unit = 'Angstrom'
        cell.basis = basis_name
        cell.pseudo = pseudo_name
        cell.verbose = 0
        cell.build()

        kpts = cell.make_kpts(self.kpts)

        # CPU KUKS → to_gpu() (без density_fit — он ломает типы)
        from pyscf.pbc import dft
        mf = dft.KUKS(cell, kpts)
        mf.xc = self.xc
        mf.direct_scf = True
        mf.max_memory = 3000
        mf.fermi_smearing = 0.02
        mf.grids.level = -3

        # Перенос на GPU
        mf = mf.to_gpu()

        # Спин через nelec
        n_elec = int(cell.nelectron)
        mf.nelec = ((n_elec + 21) // 2, n_elec - (n_elec + 21) // 2)

        print("\nЗапуск SCF на GPU...")
        mf.kernel()
        print(f"  SCF converged: {mf.converged} (циклов: {mf.cycle})")

        self.results['energy'] = mf.e_tot * HARTREE_TO_EV

        grad_method = mf.nuc_grad_method()
        forces_au = -grad_method.kernel()
        self.results['forces'] = forces_au * (HARTREE_TO_EV / BOHR_TO_ANG)

        try:
            stress_method = mf.PBCStress()
            stress_au = -stress_method.kernel()
            stress_ev = stress_au * HA_BOHR3_TO_EV_ANG3
            s = stress_ev
            self.results['stress'] = np.array([s[0,0], s[1,1], s[2,2], s[1,2], s[0,2], s[0,1]])
        except Exception as e:
            print(f"Warning: Stress failed ({e})")
            self.results['stress'] = np.zeros(6)

atoms = read('HEA_L12_SQS_32atoms.cif')
print(f"Загружено {len(atoms)} атомов.")

calc = PySCFGPUCalculator(xc='PBE', kpts=(1, 1, 1))
atoms.calc = calc
print("Калькулятор готов (gpu4pyscf GPU).")

In [ ]:
e_filter = CellFilter(atoms)
opt = BFGS(e_filter)

print("Начало релаксации на GPU...")
opt.run(fmax=0.02)

print("\nРелаксация завершена!")

In [ ]:
print(f"Финальная энергия: {atoms.get_potential_energy():.4f} eV")
print(f"\nФинальная ячейка (Angstrom):")
print(atoms.get_cell())

write('HEA_L12_relaxed_gpu.cif', atoms)
print("\nСохранен: HEA_L12_relaxed_gpu.cif")